# Cohort Analysis

## Overview
**Cohort analysis** groups users by a shared characteristic at a point in time -- usually their acquisition date -- and tracks their behaviour over subsequent periods. It answers: *are newer customers behaving better or worse than older ones?*

**Common cohort definitions:**

| Type | Definition | Example |
|---|---|---|
| Acquisition cohort | Grouped by when they first joined | Q1 2023 signups |
| Behavioural cohort | Grouped by an action they took | Customers who made a deposit in month 1 |
| Size cohort | Grouped by an attribute at acquisition | High-value vs standard customers |

**Standard cohort retention table structure:**

```
Cohort     | Size | Month 0 | Month 1 | Month 2 | Month 3 ...
-----------|------|---------|---------|---------|--------
2023-Q1    |  7   | 100%    |  71%    |  57%    |  43%
2023-Q2    |  5   | 100%    |  60%    |  40%    |  ...
```

- **Month 0** = the cohort's acquisition period (always 100%)
- **Month N** = % of the cohort still active N months later
- Reading across a row shows a cohort's retention over time
- Reading down a column shows how cohort quality changes across acquisition periods

**PostgreSQL notes:** `DATE_TRUNC('month', date)` truncates to month start. `EXTRACT(YEAR FROM AGE(d1, d2))` computes elapsed years. In SQLite, use `STRFTIME('%Y-%m', date)` and `CAST((JULIANDAY(d1) - JULIANDAY(d2))/30 AS INTEGER)`.

---

In [ ]:
import sqlite3
import pandas as pd
conn = sqlite3.connect(":memory:")
conn.executescript("""CREATE TABLE customers (customer_id INTEGER PRIMARY KEY, first_name TEXT, last_name TEXT,segment TEXT, province TEXT, signup_date TEXT, active INTEGER DEFAULT 1);CREATE TABLE accounts (account_id INTEGER PRIMARY KEY, customer_id INTEGER, account_type TEXT,opened_date TEXT, status TEXT, credit_limit REAL);CREATE TABLE transactions (txn_id INTEGER PRIMARY KEY, account_id INTEGER, customer_id INTEGER,txn_date TEXT, txn_type TEXT, amount REAL, category TEXT);CREATE TABLE product_events (event_id INTEGER PRIMARY KEY, customer_id INTEGER,event_date TEXT, event_type TEXT, product TEXT, channel TEXT);INSERT INTO customers VALUES(1,'Aroha','Ngata','Retail','NB','2023-01-05',1),(2,'Liam','Chen','Wealth','NS','2023-01-12',1),(3,'Fatima','Al-Rashid','SME','ON','2023-01-20',1),(4,'James','MacLeod','Retail','NB','2023-02-03',1),(5,'Sofia','Petrov','Retail','BC','2023-02-14',1),(6,'Noah','Williams','SME','AB','2023-02-28',0),(7,'Mei','Zhang','Wealth','ON','2023-03-10',1),(8,'Ethan','Tremblay','Retail','QC','2023-04-05',1),(9,'Priya','Nair','Wealth','BC','2023-04-18',1),(10,'Marcus','Okafor','SME','ON','2023-04-25',1),(11,'Diana','Flores','Retail','NB','2023-05-02',1),(12,'Ravi','Patel','Retail','ON','2023-05-15',1),(13,'Yuki','Tanaka','Wealth','BC','2023-06-01',1),(14,'Omar','Hassan','SME','QC','2023-06-20',1),(15,'Chloe','Bouchard','Retail','QC','2023-07-08',0);INSERT INTO accounts VALUES(101,1,'Chequing','2023-01-05','Active',NULL),(102,1,'Savings','2023-01-05','Active',NULL),(103,2,'Investment','2023-01-12','Active',50000),(104,3,'Chequing','2023-01-20','Active',NULL),(105,3,'Loan','2023-01-20','Active',25000),(106,4,'Chequing','2023-02-03','Active',NULL),(107,5,'Chequing','2023-02-14','Active',NULL),(108,5,'Savings','2023-02-14','Active',NULL),(109,6,'Chequing','2023-02-28','Closed',NULL),(110,7,'Investment','2023-03-10','Active',75000),(111,8,'Chequing','2023-04-05','Active',NULL),(112,9,'Investment','2023-04-18','Active',100000),(113,10,'Chequing','2023-04-25','Active',NULL),(114,10,'Loan','2023-04-25','Active',15000),(115,11,'Chequing','2023-05-02','Active',NULL),(116,12,'Chequing','2023-05-15','Active',NULL),(117,12,'Savings','2023-05-15','Active',NULL),(118,13,'Investment','2023-06-01','Active',60000),(119,14,'Chequing','2023-06-20','Active',NULL),(120,15,'Chequing','2023-07-08','Closed',NULL);INSERT INTO transactions VALUES(1001,101,1,'2023-01-10','Deposit',2500.00,'Payroll'),(1002,101,1,'2023-01-22','Withdrawal',-800.00,'Rent'),(1003,102,1,'2023-02-05','Deposit',500.00,'Transfer'),(1004,103,2,'2023-02-10','Deposit',10000.00,'Investment'),(1005,104,3,'2023-02-15','Deposit',3200.00,'Payroll'),(1006,104,3,'2023-03-01','Withdrawal',-1200.00,'Rent'),(1007,106,4,'2023-03-10','Deposit',2800.00,'Payroll'),(1008,107,5,'2023-03-15','Deposit',2200.00,'Payroll'),(1009,107,5,'2023-03-28','Fee',-25.00,'Monthly Fee'),(1010,101,1,'2023-03-10','Deposit',2500.00,'Payroll'),(1011,103,2,'2023-04-15','Deposit',15000.00,'Investment'),(1012,110,7,'2023-04-20','Deposit',20000.00,'Investment'),(1013,111,8,'2023-05-01','Deposit',2100.00,'Payroll'),(1014,112,9,'2023-05-10','Deposit',25000.00,'Investment'),(1015,113,10,'2023-05-20','Deposit',3500.00,'Payroll'),(1016,115,11,'2023-06-01','Deposit',2000.00,'Payroll'),(1017,116,12,'2023-06-15','Deposit',2700.00,'Payroll'),(1018,118,13,'2023-07-01','Deposit',18000.00,'Investment'),(1019,101,1,'2023-07-10','Deposit',2500.00,'Payroll'),(1020,103,2,'2023-07-20','Deposit',12000.00,'Investment'),(1021,104,3,'2023-07-15','Deposit',3200.00,'Payroll'),(1022,107,5,'2023-08-01','Deposit',2200.00,'Payroll'),(1023,111,8,'2023-08-05','Deposit',2100.00,'Payroll'),(1024,113,10,'2023-08-15','Withdrawal',-500.00,'Transfer'),(1025,116,12,'2023-08-20','Deposit',2700.00,'Payroll'),(1026,101,1,'2023-10-10','Deposit',2500.00,'Payroll'),(1027,116,12,'2023-10-20','Deposit',2700.00,'Payroll');INSERT INTO product_events VALUES(1,1,'2023-01-10','page_view','Savings Account','web'),(2,1,'2023-01-10','product_view','Savings Account','web'),(3,1,'2023-01-10','application_start','Savings Account','web'),(4,1,'2023-01-11','application_submit','Savings Account','web'),(5,1,'2023-01-15','account_opened','Savings Account','web'),(6,2,'2023-01-12','page_view','Investment Account','mobile'),(7,2,'2023-01-12','product_view','Investment Account','mobile'),(8,2,'2023-01-12','application_start','Investment Account','mobile'),(9,2,'2023-01-13','application_submit','Investment Account','mobile'),(10,2,'2023-01-15','account_opened','Investment Account','mobile'),(11,3,'2023-01-20','page_view','Chequing Account','web'),(12,3,'2023-01-20','product_view','Chequing Account','web'),(13,3,'2023-01-20','application_start','Chequing Account','web'),(14,4,'2023-02-03','page_view','Chequing Account','web'),(15,4,'2023-02-03','product_view','Chequing Account','web'),(16,5,'2023-02-14','page_view','Savings Account','mobile'),(17,5,'2023-02-14','product_view','Savings Account','mobile'),(18,5,'2023-02-14','application_start','Savings Account','mobile'),(19,5,'2023-02-15','application_submit','Savings Account','mobile'),(20,5,'2023-02-20','account_opened','Savings Account','mobile'),(21,7,'2023-03-10','page_view','Investment Account','web'),(22,7,'2023-03-10','product_view','Investment Account','web'),(23,7,'2023-03-10','application_start','Investment Account','web'),(24,7,'2023-03-11','application_submit','Investment Account','web'),(25,7,'2023-03-15','account_opened','Investment Account','web'),(26,8,'2023-04-05','page_view','Chequing Account','mobile'),(27,8,'2023-04-05','product_view','Chequing Account','mobile'),(28,9,'2023-04-18','page_view','Investment Account','web'),(29,9,'2023-04-18','product_view','Investment Account','web'),(30,9,'2023-04-18','application_start','Investment Account','web'),(31,9,'2023-04-20','application_submit','Investment Account','web'),(32,9,'2023-04-25','account_opened','Investment Account','web');""")
conn.commit()
print("Finance schema ready.")
for t in ["customers","accounts","transactions","product_events"]:
    n = conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"  {t}: {n} rows")

---
## Defining cohorts by signup quarter

In [ ]:
print("=== Assign each customer to a signup cohort (quarter) ===")
q = """
SELECT  customer_id,
        first_name || ' ' || last_name             AS customer,
        segment,
        signup_date,
        CASE
            WHEN STRFTIME('%m', signup_date) IN ('01','02','03') THEN 'Q1-2023'
            WHEN STRFTIME('%m', signup_date) IN ('04','05','06') THEN 'Q2-2023'
            ELSE                                                       'Q3-2023'
        END  AS cohort
FROM    customers
ORDER BY signup_date
"""
df = pd.read_sql(q, conn)
print(df.to_string(index=False))
print()
print("Cohort sizes:")
print(df.groupby('cohort').size().reset_index(name='customers').to_string(index=False))


---
## Building a monthly retention table

In [ ]:
print("=== Monthly cohort retention: which customers transacted in each month? ===")
# Step 1: assign cohort to each customer
# Step 2: find all months each customer had a transaction
# Step 3: compute months_since_signup
# Step 4: pivot into cohort x period retention table

q = """
WITH cohorts AS (
    SELECT  customer_id,
            CASE
                WHEN STRFTIME('%m', signup_date) IN ('01','02','03') THEN 'Q1-2023'
                WHEN STRFTIME('%m', signup_date) IN ('04','05','06') THEN 'Q2-2023'
                ELSE 'Q3-2023'
            END  AS cohort,
            signup_date
    FROM    customers
),
active_months AS (
    SELECT  DISTINCT
            t.customer_id,
            STRFTIME('%Y-%m', t.txn_date)  AS active_month
    FROM    transactions AS t
),
cohort_activity AS (
    SELECT  c.cohort,
            c.customer_id,
            a.active_month,
            -- months since signup (approximate: floor(days/30))
            CAST((JULIANDAY(a.active_month || '-01') -
                  JULIANDAY(c.signup_date)) / 30 AS INTEGER)  AS months_since_signup
    FROM    cohorts       AS c
    JOIN    active_months AS a ON c.customer_id = a.customer_id
),
cohort_sizes AS (
    SELECT  cohort, COUNT(*) AS cohort_size
    FROM    cohorts GROUP BY cohort
),
retention AS (
    SELECT  ca.cohort,
            ca.months_since_signup,
            COUNT(DISTINCT ca.customer_id)  AS retained_customers
    FROM    cohort_activity AS ca
    WHERE   ca.months_since_signup BETWEEN 0 AND 6
    GROUP BY ca.cohort, ca.months_since_signup
)
SELECT  r.cohort,
        cs.cohort_size,
        r.months_since_signup AS month,
        r.retained_customers,
        ROUND(100.0 * r.retained_customers / cs.cohort_size, 1) AS retention_pct
FROM    retention AS r
JOIN    cohort_sizes AS cs ON r.cohort = cs.cohort
ORDER BY r.cohort, r.months_since_signup
"""
df = pd.read_sql(q, conn)
print(df.to_string(index=False))


---
## Pivot: wide retention table

In [ ]:
print("=== Cohort retention table (wide format) ===")
q = """
WITH cohorts AS (
    SELECT customer_id,
           CASE WHEN STRFTIME('%m',signup_date) IN ('01','02','03') THEN 'Q1-2023'
                WHEN STRFTIME('%m',signup_date) IN ('04','05','06') THEN 'Q2-2023'
                ELSE 'Q3-2023' END AS cohort,
           signup_date
    FROM customers
),
active_months AS (
    SELECT DISTINCT customer_id, STRFTIME('%Y-%m', txn_date) AS active_month
    FROM transactions
),
retention AS (
    SELECT c.cohort, cs.cohort_size, a.customer_id,
           CAST((JULIANDAY(a.active_month||'-01') - JULIANDAY(c.signup_date))/30 AS INTEGER) AS m
    FROM cohorts c
    JOIN active_months a ON c.customer_id = a.customer_id
    JOIN (SELECT cohort, COUNT(*) cohort_size FROM cohorts GROUP BY cohort) cs ON c.cohort = cs.cohort
    WHERE CAST((JULIANDAY(a.active_month||'-01') - JULIANDAY(c.signup_date))/30 AS INTEGER) BETWEEN 0 AND 5
)
SELECT  cohort,
        MAX(cohort_size)  AS size,
        COUNT(DISTINCT CASE WHEN m=0 THEN customer_id END) AS m0,
        COUNT(DISTINCT CASE WHEN m=1 THEN customer_id END) AS m1,
        COUNT(DISTINCT CASE WHEN m=2 THEN customer_id END) AS m2,
        COUNT(DISTINCT CASE WHEN m=3 THEN customer_id END) AS m3,
        COUNT(DISTINCT CASE WHEN m=4 THEN customer_id END) AS m4,
        COUNT(DISTINCT CASE WHEN m=5 THEN customer_id END) AS m5
FROM    retention
GROUP BY cohort
ORDER BY cohort
"""
df = pd.read_sql(q, conn)

# Show as percentages
for col in ['m0','m1','m2','m3','m4','m5']:
    df[col+'_pct'] = (100.0 * df[col] / df['size']).round(1).astype(str) + '%'

print("Raw counts:")
print(df[['cohort','size','m0','m1','m2','m3','m4','m5']].to_string(index=False))
print()
print("As retention %:")
print(df[['cohort','size','m0_pct','m1_pct','m2_pct','m3_pct','m4_pct','m5_pct']].to_string(index=False))
conn.close()


---
## Common Pitfalls

**1. Using calendar periods instead of cohort-relative periods**
Comparing "Q1 cohort in March" to "Q2 cohort in March" mixes cohort age. Month 0 must mean the period of acquisition for each cohort, and Month N means N periods later -- not the same calendar month. Always compute periods relative to each cohort's acquisition date.

**2. Counting users who appeared in any transaction, not necessarily recurring**
A customer who signed up and made one deposit in month 0, then never returned, shows 100% in month 0 and 0% afterwards. That is correct. But if you count "ever active" rather than "active in this period", retention looks inflated. Use `COUNT(DISTINCT customer_id)` within each specific period.

**3. Cohort size denominator drift**
If some customers in the Q1 cohort churned before you could observe month 3, the denominator for month 3 should still be the full Q1 cohort size -- not just those who survived to month 3. Shrinking the denominator makes retention look better than it is.

**4. Not accounting for cohort maturity**
A Q3 cohort can only have month 0 and month 1 data if the analysis runs in Q4. Comparing Q1's month 6 retention to Q3's month 6 (which doesn't exist yet) produces NULL or misleading zeros. Always mark immature cohort cells as NULL or N/A rather than 0.

**5. Month approximation errors with JULIANDAY/30**
Dividing Julian day difference by 30 is an approximation. January has 31 days; February 28. A customer who signed up January 31 and transacted March 1 might show as month 1 or month 2 depending on the approximation. For precise cohort periods, use `DATE_TRUNC('month', ...)` in PostgreSQL or `STRFTIME('%Y-%m', ...)` comparisons in SQLite.

**6. Confusing retention with revenue retention**
Customer count retention and revenue retention tell different stories. A cohort that retains 50% of customers but those customers triple their spending has strong revenue retention despite weak count retention. Report both when customer lifetime value matters.


---
*sql_methods_library - Samantha McGarrigle*